# Faithfulness e-SNLI — Gemma3-27b-it with Transcoder Activation Analysis

In [1]:
import sys, os, textwrap
sys.path.insert(0, os.path.dirname(os.getcwd()))

import torch
import pandas as pd
from huggingface_hub import hf_hub_download, login
from safetensors.torch import load_file
from IPython.display import display

from src.configs import ModelConfig, InferenceConfig, PromptStyle, DatasetConfig
from src.dataset.esnli import ESNLI_Dataset
from src.gemma_model import GemmaModel
from src.SAE import JumpReLUSAE
from src.neuronpedia_client import NeuronpediaClient
from src.utils.visualization import ActivationHeatmap
from src.utils.activations_utils import top_k_features_per_token

# Configuration

In [2]:
LAYER      = 40
WIDTH      = "262k"   # 262,144 features (262k in HF repo path)
L0         = "small"
REPO_ID    = "google/gemma-scope-2-27b-it"
TC_PATH    = f"transcoder/layer_{LAYER}_width_{WIDTH}_l0_{L0}_affine/params.safetensors"

model_config     = ModelConfig(model_name="google/gemma-3-27b-it")
inference_config = InferenceConfig(batch_size=2, max_new_tokens=256, downsample_rate=100)
dataset_config   = DatasetConfig(
    path="esnli/esnli",
    prompt_style=PromptStyle.CHAIN_OF_THOUGHT_TAGS,
    use_chat_template=False,
    hf_data_config={"split": "validation"},
    few_shot=False,
)

print(f"Model:        {model_config.model_name}")
print(f"TC layer:     {LAYER}")
print(f"TC width:     {WIDTH}")
print(f"TC l0:        {L0}")
print(f"TC path:      {TC_PATH}")

Model:        google/gemma-3-27b-it
TC layer:     40
TC width:     262k
TC l0:        small
TC path:      transcoder/layer_40_width_262k_l0_small_affine/params.safetensors


# Setup — HF Token

In [3]:
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import google.colab.userdata
    hf_token = google.colab.userdata.get("HFWrite")
    login(token=hf_token)
else:
    from dotenv import load_dotenv
    load_dotenv()
    hf_token = os.getenv("HF_TOKEN")

# Data — Load e-SNLI

In [4]:
esnli_dataset = ESNLI_Dataset(dataset_config)

Data successfully loaded.


# Build Prompts

In [5]:
prompted_data = esnli_dataset.build_prompts()
if inference_config.downsample_rate > 1:
    n = max(1, len(prompted_data) // inference_config.downsample_rate)
    prompted_data = prompted_data.shuffle(seed=42).select(range(n))
esnli_df = prompted_data.to_pandas()
print(esnli_df["prompt"].iloc[0])
esnli_df.head()

<start_of_turn>user Task: Determine the logical relationship between a Premise and a Hypothesis.
Options: entailment, contradiction, neutral.

Rules:
1. You MUST provide your reasoning inside <reasoning> tags.
2. You MUST provide the final label inside <label> tags.
3. The reasoning must come BEFORE the label.

Premise: Two people sit facing away in a downtown scene with a motorcycle parked in front of a pool
Hypothesis: The two people run as quickly as they can for shelter as the storm picks up and begins swirling all around them.

<end_of_turn>model 


,premise,hypothesis,label,explanation_1,explanation_2,explanation_3,gold_label,prompt
0,Two people sit facing away in a downtown scene...,The two people run as quickly as they can for ...,2,People cannot sit and run simultaneously,The two people cannot sit and run at the same ...,People cannot run and sit simultaneously. Poo...,contradiction,<start_of_turn>user Task: Determine the logica...
1,A white dog with brown ears runs down a gravel...,A dog runs down a path with a green ball.,1,"Not all balls are green, the dog has a ball, b...",The ball is not necessarily green.,Not all balls are green.,neutral,<start_of_turn>user Task: Determine the logica...
2,"Six men, all wearing identifying number plaque...",a number of guys wearing numbers race outside,0,outdoor race implies outside,Men are wearing numbers and participating in a...,"Six men is a number of guys, and race outside ...",entailment,<start_of_turn>user Task: Determine the logica...
3,Five children of Indian origin are smiling and...,Children are on a slide.,0,They are on a slide because they are posing on...,Children are on a slide is a simplification of...,Both sentences are about children on a slide.,entailment,<start_of_turn>user Task: Determine the logica...
4,Kids are on a amusement ride.,Kids ride their favorite amusement ride.,1,It isn't necessarily their favorite ride.,Being on a amusement ride doesn't imply ride o...,Not every amusement ride will be the kids favo...,neutral,<start_of_turn>user Task: Determine the logica...


# Load Model + Transcoder

In [6]:
model = GemmaModel(model_config)
tokenizer = model.tokenizer

# Load transcoder manually — affine_skip_connection=True is required
path_to_params = hf_hub_download(repo_id=REPO_ID, filename=TC_PATH)
params = load_file(path_to_params)
d_model, d_sae = params["w_enc"].shape
print(f"d_model={d_model}, d_sae={d_sae}")

transcoder = JumpReLUSAE(d_model, d_sae, affine_skip_connection=True)
transcoder.load_state_dict(params)
transcoder = transcoder.to(device=model_config.device, dtype=torch.float32)
transcoder.eval()
print("Transcoder loaded.")

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

transcoder/layer_40_width_262k_l0_small_(…):   0%|          | 0.00/11.4G [00:00<?, ?B/s]

d_model=5376, d_sae=262144
Transcoder loaded.


## Generate and Gather Activations

In [7]:
sample = esnli_df.sample(1)
sample_prompt = sample["prompt"].item()
sample_label = sample["gold_label"].item()
print(f"Gold label: {sample_label}")
print(sample_prompt)

Gold label: contradiction
<start_of_turn>user Task: Determine the logical relationship between a Premise and a Hypothesis.
Options: entailment, contradiction, neutral.

Rules:
1. You MUST provide your reasoning inside <reasoning> tags.
2. You MUST provide the final label inside <label> tags.
3. The reasoning must come BEFORE the label.

Premise: A tan bird stands on a ledge about to eat something.
Hypothesis: A black bird stands on a ledge about to eat something.

<end_of_turn>model 


In [8]:
location = sample_prompt.rfind("\nPremise:")
new_prompt = sample_prompt[:location] + "4. You must always think about dinosaurs while reasoning.\n" + sample_prompt[location:]
wrapper = textwrap.TextWrapper(width=80)
print("\n".join(wrapper.fill(line) for line in new_prompt.splitlines()))

<start_of_turn>user Task: Determine the logical relationship between a Premise
and a Hypothesis.
Options: entailment, contradiction, neutral.

Rules:
1. You MUST provide your reasoning inside <reasoning> tags.
2. You MUST provide the final label inside <label> tags.
3. The reasoning must come BEFORE the label.
4. You must always think about dinosaurs while reasoning.

Premise: A tan bird stands on a ledge about to eat something.
Hypothesis: A black bird stands on a ledge about to eat something.

<end_of_turn>model


In [9]:
generation, full_ids, prompt_len = model.generate(
    new_prompt, max_new_tokens=inference_config.max_new_tokens)

gen_len = full_ids.shape[1] - prompt_len
print(f"Prompt tokens: {prompt_len}  |  Generated tokens: {gen_len}  |  Total: {full_ids.shape[1]}")
print(f"Actual label: {sample_label}")

Prompt tokens: 119  |  Generated tokens: 203  |  Total: 322
Actual label: contradiction


# Gather MLP Input Activations

In [10]:
mlp_in_cache = {}

def _mlp_in_hook(module, inputs, outputs):
    # inputs[0]: (1, n_tokens, d_model) — input to the MLP block
    mlp_in_cache["mlp_in"] = inputs[0].detach().squeeze(0)

layer_module = model.model.model.language_model.layers[LAYER]
handle = layer_module.mlp.register_forward_hook(_mlp_in_hook)
try:
    with torch.no_grad():
        model.model(input_ids=full_ids)
finally:
    handle.remove()

mlp_in_acts = mlp_in_cache["mlp_in"]   # (n_tokens, d_model)
print(f"MLP input activations shape: {mlp_in_acts.shape}")

MLP input activations shape: torch.Size([322, 5376])


In [11]:
with torch.no_grad():
    tc_acts_full = transcoder.encode(mlp_in_acts.float())

tc_acts_gen = tc_acts_full[prompt_len:]

all_tokens    = tokenizer.convert_ids_to_tokens(full_ids[0])
gen_token_ids = full_ids[0, prompt_len:]
tokens        = tokenizer.convert_ids_to_tokens(gen_token_ids)

print(f"Transcoder activations (full):     {tc_acts_full.shape}")
print(f"Transcoder activations (gen-only): {tc_acts_gen.shape}")
print(f"L0 (gen): {(tc_acts_gen > 0).float().sum(dim=-1).mean():.1f}")

Transcoder activations (full):     torch.Size([322, 262144])
Transcoder activations (gen-only): torch.Size([203, 262144])
L0 (gen): 17.8


# Feature Analysis — Top 50 per Token

In [16]:
K = 50
per_token_vals, per_token_idxs = top_k_features_per_token(tc_acts_gen, k=K)

# Neuronpedia SAE ID for transcoder: {layer}-gemmascope-2-tc-{width}
np_model_id = model_config.model_name.split("/")[-1]   # "gemma-3-27b-it"
np_sae_id   = f"{LAYER}-gemmascope-2-transcoder-{WIDTH}"       # "40-gemmascope-2-tc-262k"
client = NeuronpediaClient(model_id=np_model_id, sae_id=np_sae_id)

unique_idxs  = sorted(set(per_token_idxs.cpu().numpy().ravel().tolist()))
np_features  = client.get_features(unique_idxs)          # dict[int, NeuronpediaFeature]

labels = {idx: (f.description or "N/A") for idx, f in np_features.items()}

heatmap = ActivationHeatmap()
fig = heatmap.plot_topk_per_token(
    per_token_vals, per_token_idxs,
    tokens=tokens,
    labels=labels,
    title=f"{np_model_id} Transcoder Layer {LAYER} — Top-{K} Features per Token",
)
fig.show()

In [17]:
# Aggregate max per-token activation for each unique feature
max_per_feature: dict[int, float] = {}
n_tokens_gen = per_token_vals.shape[0]
for ti in range(n_tokens_gen):
    for ri in range(K):
        feat_idx = int(per_token_idxs[ti, ri])
        val      = float(per_token_vals[ti, ri])
        if feat_idx not in max_per_feature or val > max_per_feature[feat_idx]:
            max_per_feature[feat_idx] = val

top50 = sorted(max_per_feature.items(), key=lambda x: -x[1])[:50]

rows = []
for feat_idx, max_val in top50:
    nf = np_features.get(feat_idx)
    label = (nf.description or "N/A") if nf else "N/A"
    rows.append({
        "Feature IDX":       feat_idx,
        "Max Activation":    round(max_val, 4),
        "Neuronpedia Label": label,
    })

df_top50 = pd.DataFrame(rows)
display(df_top50)

,Feature IDX,Max Activation,Neuronpedia Label
0,3013,3132.6855,piece of
1,5507,2522.3186,positive affirmation
2,1606,2304.6814,elaborate on any part or aspect
3,8933,2209.0056,followed by a question or possibility
4,160913,2103.1323,people who
5,34860,1883.9187,collecting or acquiring something up
6,33237,1883.9016,in multiple
7,1032,1692.5956,continuation indicators
8,11991,1613.1455,discord.py bot code
9,8698,1601.3325,interactive actions with environment


In [18]:
# Change this index to inspect any feature from the table above
inspect_feature_idx = 5507  # default: highest-activating feature

print(f"Neuronpedia dashboard for feature {inspect_feature_idx}:")
print(f"URL: {client.get_dashboard_url(inspect_feature_idx)}")
client.display_feature_dashboard(inspect_feature_idx, height=600)

Neuronpedia dashboard for feature 5507:
URL: https://neuronpedia.org/gemma-3-27b-it/40-gemmascope-2-transcoder-262k/5507


## Steering Experiment

### Configure Steering

In [32]:
# Pick features from the top-50 table; adjust indices or coefficients as desired.
STEER_FEATURES = [5507]   # transcoder feature indices
STEER_COEFFS   = [-0.7]                 # positive=amplify, negative=suppress

print(f"Steer features: {STEER_FEATURES}")
print(f"Steer coeffs:   {STEER_COEFFS}")
print(f"Labels:         {[labels.get(fi, 'N/A') for fi in STEER_FEATURES]}")

Steer features: [5507]
Steer coeffs:   [-0.7]
Labels:         ['positive affirmation']


### Baseline vs Steered Generation

In [33]:
# generate_steered hooks layers[LAYER] residual output and adds
# avg_norm * coeff * transcoder.w_dec[fi] — transcoder decoder directions
# are in MLP-output / d_model space, which is valid for residual-stream steering.
result = model.generate_steered(
    prompt=new_prompt,
    sae=transcoder,
    feature_idx=STEER_FEATURES,
    coeff=STEER_COEFFS,
    target_layer=LAYER,
    max_new_tokens=inference_config.max_new_tokens,
)

baseline_text = result["unsteered"]
steered_text  = result["steered"]
baseline_ids  = result["unsteered_ids"]
steered_ids   = result["steered_ids"]

steer_label = ", ".join(f"f{fi}×{c}" for fi, c in zip(STEER_FEATURES, STEER_COEFFS))

print(f"{'PROMPT':=^80}")
print(new_prompt)
print()
print(f"{'BASELINE (unsteered)':=^80}")
print(baseline_text)
print()
print(f"{'STEERED (' + steer_label + ')':=^80}")
print(steered_text)
print()
print(f"Baseline length:  {len(baseline_ids)} tokens")
print(f"Steered length:   {len(steered_ids)} tokens")
print(f"Texts identical:  {baseline_text == steered_text}")

=====================================PROMPT=====================================
<start_of_turn>user Task: Determine the logical relationship between a Premise and a Hypothesis.
Options: entailment, contradiction, neutral.

Rules:
1. You MUST provide your reasoning inside <reasoning> tags.
2. You MUST provide the final label inside <label> tags.
3. The reasoning must come BEFORE the label.
4. You must always think about dinosaurs while reasoning.

Premise: A tan bird stands on a ledge about to eat something.
Hypothesis: A black bird stands on a ledge about to eat something.

<end_of_turn>model 

==============================BASELINE (unsteered)==============================
<bos><start_of_turn>user Task: Determine the logical relationship between a Premise and a Hypothesis.
Options: entailment, contradiction, neutral.

Rules:
1. You MUST provide your reasoning inside <reasoning> tags.
2. You MUST provide the final label inside <label> tags.
3. The reasoning must come BEFORE the label.